In [1]:
import time 
import warnings 
from pathlib import Path 

import torch 
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B,
    KVCache,
)
from reasoning_from_scratch.qwen3 import download_qwen3_small 
download_qwen3_small(kind = 'base', tokenizer_only = True, out_dir = 'qwen3')

from reasoning_from_scratch.qwen3 import Qwen3Tokenizer 

In [2]:
print(f'PyTorch version {torch.__version__}')

if torch.backends.mps.is_available():
    print('MPS')
else:
    print('CPU')

PyTorch version 2.10.0
MPS


In [3]:
tokenizer_path = Path('qwen3/tokenizer-base.json')
tokenizer = Qwen3Tokenizer(tokenizer_file_path = tokenizer_path)

In [4]:
prompt = 'andromeda' 
input_token_ids_list = tokenizer.encode(prompt)

text = tokenizer.decode(input_token_ids_list)
print(text)

andromeda


In [5]:
input_token_ids_list

[437, 441, 13830]

In [6]:
for i in input_token_ids_list:
    print(f'{i} --> {tokenizer.decode([i])}')

437 --> and
441 --> rom
13830 --> eda


In [7]:
def get_device():
    if torch.backends.mps.is_available():
        device = torch.device('mps') 
    else:
        device = torch.device('cpu')
    return device 

device = get_device()
print(device)

mps


In [8]:
download_qwen3_small(kind = 'base', tokenizer_only = False, out_dir = 'qwen3')

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [9]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B 

model_path = Path('qwen3/qwen3-0.6B-base.pth')
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))
model.to(device) 

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [10]:
prompt = 'Explain large language models.' 
input_token_ids_list = tokenizer.encode(prompt)
print(f'number of input tokens: {len(input_token_ids_list)}')

input_tensor = torch.tensor(input_token_ids_list)
input_tensor_fmt = input_tensor.unsqueeze(0)
input_tensor_fmt = input_tensor_fmt.to(device)

with torch.inference_mode():            # offs the
    output_tensor = model(input_tensor_fmt)

output_tensor_fmt = output_tensor.squeeze(0)
print(f'formatted output tensor shape: {output_tensor_fmt.shape}')
output_tensor_fmt

number of input tokens: 6
formatted output tensor shape: torch.Size([6, 151936])


tensor([[ 7.4062, 11.4375,  9.2500,  ...,  3.7188,  3.7188,  3.7188],
        [ 9.3750, 10.5625,  7.2500,  ...,  3.2344,  3.2344,  3.2344],
        [10.8750, 10.0000,  7.5938,  ...,  0.1992,  0.1992,  0.1992],
        [ 7.1250,  9.2500,  6.2812,  ..., -1.9844, -1.9844, -1.9844],
        [11.5000, 13.6250, 10.2500,  ...,  1.0000,  1.0000,  1.0000],
        [ 7.3438,  2.0312,  7.9375,  ..., -2.5156, -2.5156, -2.5156]],
       device='mps:0', dtype=torch.bfloat16)

In [11]:
last_token = output_tensor_fmt[-1]
print(last_token)

tensor([ 7.3438,  2.0312,  7.9375,  ..., -2.5156, -2.5156, -2.5156],
       device='mps:0', dtype=torch.bfloat16)


In [12]:
print(torch.argmax(last_token, dim = -1, keepdim = True))

tensor([20286], device='mps:0')


In [13]:
print(tokenizer.decode([20286]))

 Large


In [14]:
@torch.inference_mode()

def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id = None 
):
    model.eval()
    
    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1, :]
        next_token = torch.argmax(out, dim = -1, keepdim = True)
        
        if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
            break
        yield next_token
        token_ids = torch.cat([token_ids, next_token], dim = 1)

In [15]:
prompt = 'Explain large language models in a single sentence.' 

input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device = device).unsqueeze(0)

max_new_tokens = 100 

for token in generate_text_basic_stream(
    model = model,
    token_ids = input_token_ids_tensor,
    max_new_tokens = max_new_tokens,
):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end = '', flush = True)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays, and even creating creative content.<|endoftext|>Human language is a complex and dynamic system that has evolved over millions of years to enable effective communication and social interaction. It is composed of a vast array of symbols, including letters, numbers, and words, which are used to convey meaning and express thoughts and ideas. The structure of human language

In [16]:
print(tokenizer.encode('<|endoftext|>'))

[151643]


In [17]:
for token in generate_text_basic_stream(
    model = model,
    token_ids = input_token_ids_tensor,
    max_new_tokens = max_new_tokens,
    eos_token_id = tokenizer.eos_token_id 
):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end = '', flush = True)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays, and even creating creative content.

In [18]:
def generate_stats(output_token_ids, tokenizer, start_time, end_time):
    total_time = end_time - start_time 
    print(f'\n\nTime: {total_time:.2f} sec')
    print(f'{int(output_token_ids.numel() / total_time)} token/sec')
    
    if torch.backends.mps.is_available():
        torch.mps.synchronize()
        mem_bytes = torch.mps.current_allocated_memory()
        mem_gb = mem_bytes / (1024 ** 3)
        print(f'MPS memory allocated: {mem_gb:.2f} GB')



In [19]:
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model = model,
    token_ids = input_token_ids_tensor,
    max_new_tokens = max_new_tokens,
    eos_token_id = tokenizer.eos_token_id 
):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end = '', flush = True)
    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)

end_time = time.time()
output_token_ids_tensor = torch.cat(generated_ids, dim = 0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays, and even creating creative content.

Time: 1.86 sec
22 token/sec
MPS memory allocated: 1.44 GB


In [20]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()

def generate_text_basic_stream_cache(
    model, 
    token_ids, 
    max_new_tokens,
    eos_token_id = None 
):
    
    model.eval()
    cache = KVCache(n_layers = model.cfg['n_layers'])
    model.reset_kv_cache()
    
    out = model(token_ids, cache = cache)[:, -1]
    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim = -1, keepdim = True)
        
        if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
            break 
        
        yield next_token 
        out = model(next_token, cache = cache)[:, -1]

In [21]:
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(
    model = model,
    token_ids = input_token_ids_tensor,
    max_new_tokens = max_new_tokens,
    eos_token_id = tokenizer.eos_token_id 
):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end = '', flush = True)
    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)
    
end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim = 0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Time: 2.46 sec
16 token/sec
MPS memory allocated: 1.44 GB


In [22]:
major, minor = map(int, torch.__version__.split('.')[:2])
if (major, minor) >= (2, 8):
    torch._dynamo.config.allow_unspec_int_on_nn_module = True 

model_compiled = torch.compile(model)
# model = torch.compile(model, mode = 'max-autotune')

In [23]:
for i in range(3):
    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream(
        model = model,
        token_ids = input_token_ids_tensor,
        max_new_tokens = max_new_tokens,
        eos_token_id = tokenizer.eos_token_id 
    ):
        token_id = token.squeeze(0).tolist()
        print(tokenizer.decode(token_id), end = '', flush = True)
        
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)
        
    end_time = time.time()
    
    if i == 0:
        print('\n\nwarm-up run')
    else:
        print(f'\n\ntimed run {i}')
        
    output_token_ids_tensor = torch.cat(generated_ids, dim = 0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)
    
    print(f'\n{30*'-'}\n')

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays, and even creating creative content.

warm-up run


Time: 1.86 sec
22 token/sec
MPS memory allocated: 1.44 GB

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays, and even creating creative content.

timed run 1


Time: 1.96 sec
20 token/sec
MPS memory allocated: 1.44 GB

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays, and even creating creative content.

timed run 2


Time: 1.95 sec
21 token/sec
MPS mem

In [24]:
for i in range(3):
    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream_cache(
        model = model,
        token_ids = input_token_ids_tensor,
        max_new_tokens = max_new_tokens,
        eos_token_id = tokenizer.eos_token_id 
    ):
        token_id = token.squeeze(0).tolist()
        print(tokenizer.decode(token_id), end = '', flush = True)
        
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)
        
    end_time = time.time()
    
    if i == 0:
        print('\n\nwarm-up run')
    else:
        print(f'\n\ntimed run')
        
    output_token_ids_tensor = torch.cat(generated_ids, dim = 0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)
    
    print(f'\n{30*'-'}\n')

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

warm-up run


Time: 1.32 sec
30 token/sec
MPS memory allocated: 1.44 GB

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

timed run


Time: 1.28 sec
31 token/sec
MPS memory allocated: 1.44 GB

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

timed run


Time: 1.31 sec
31 token/sec
MPS m